# Qwen2.5 SFT all models (one run ID)


In [ ]:
import os, subprocess, sys
from pathlib import Path

LAUNCH_DIR = Path.cwd().resolve()
subprocess.run([sys.executable, "-m", "pip", "install", "python-dotenv>=1,<2"], check=True)
from dotenv import load_dotenv

ENV_FILE = Path(os.environ.get("CRASHDIAG_ENV_FILE", LAUNCH_DIR / "env.txt")).expanduser()
if not ENV_FILE.is_absolute():
    ENV_FILE = (LAUNCH_DIR / ENV_FILE).resolve()
if not ENV_FILE.is_file():
    raise RuntimeError(f"CrashDiag env file not found: {ENV_FILE}")
load_dotenv(ENV_FILE, override=True)

os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

REPO_URL = os.environ.get("CRASHDIAG_REPO_URL", "https://github.com/Indium-AI-Labs/CrashDiag.git")
SOURCE_COMMIT = os.environ.get("CRASHDIAG_SOURCE_COMMIT", "main")
WORKDIR = Path(os.environ.get("CRASHDIAG_WORKDIR", LAUNCH_DIR / "CrashDiag-runtime")).expanduser().resolve()
if (WORKDIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
elif WORKDIR.exists() and any(WORKDIR.iterdir()):
    raise RuntimeError(f"CRASHDIAG_WORKDIR exists and is not a Git checkout: {WORKDIR}")
else:
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
subprocess.run(["git", "-C", str(WORKDIR), "checkout", SOURCE_COMMIT], check=True)
os.chdir(WORKDIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "bitsandbytes"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[train]"], check=True)
print(f"env_file={ENV_FILE if ENV_FILE.is_file() else 'not present (using runtime/Kaggle secrets)'}")
print("checked_out_source_commit=" + subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
import os

def ist_run_id(stage):
    return datetime.now(ZoneInfo("Asia/Kolkata")).strftime("%Y%m%dT%H%M%SIST") + f"-{stage}"
BUCKET_ID = "devaanshpa/CrashDiag"
MODELS = {
    "qwen2.5_14b": "Qwen/Qwen2.5-14B-Instruct",
    "qwen2.5_7b": "Qwen/Qwen2.5-7B-Instruct",
    "qwen2.5_3b": "Qwen/Qwen2.5-3B-Instruct",
    "qwen2.5_1.5b": "Qwen/Qwen2.5-1.5B-Instruct",
    "qwen2.5_0.5b": "Qwen/Qwen2.5-0.5B-Instruct",
}
DATASET_RUN_ID = os.environ.get("CRASHDIAG_DATASET_RUN_ID", "").strip()
ALL_RUN_ID = os.environ.get("CRASHDIAG_ALL_RUN_ID", "").strip() or ist_run_id("all")
if not DATASET_RUN_ID:
    raise RuntimeError("Set CRASHDIAG_DATASET_RUN_ID to the fresh dataset-generation run ID.")
print(f"models={list(MODELS)}")
print(f"dataset_run_id={DATASET_RUN_ID}")
print(f"ALL_RUN_ID={ALL_RUN_ID}")

In [ ]:
from pathlib import Path
from training.artifacts import ArtifactConfig, ArtifactUploader

DATASET_DIR = Path("artifacts/datasets")
ArtifactUploader(ArtifactConfig(bucket_id=BUCKET_ID, run_id=DATASET_RUN_ID, token=os.environ["HF_TOKEN"])).download_stage("datasets", DATASET_DIR)
assert (DATASET_DIR / "sft_train.jsonl").is_file(), f"dataset stage missing sft_train.jsonl; check CRASHDIAG_DATASET_RUN_ID={DATASET_RUN_ID}"
print(f"sft_train={DATASET_DIR / 'sft_train.jsonl'}")

In [ ]:
import subprocess, sys

results = {}
for slug, base_model in MODELS.items():
    output_dir = f"outputs/{slug}-sft"
    print(f"=== SFT {base_model} ({slug}) ===")
    command = [
        sys.executable, "-m", "accelerate.commands.launch",
        "--num_processes", "1", "--num_machines", "1",
        "--mixed_precision", "bf16", "--dynamo_backend", "no",
        "-m", "training.sft",
        "--model", base_model,
        "--dataset", str(DATASET_DIR / "sft_train.jsonl"),
        "--eval-dataset", str(DATASET_DIR / "sft_eval.jsonl"),
        "--output-dir", output_dir,
        "--epochs", "1",
        "--batch-size", "1",
        "--eval-batch-size", "1",
        "--gradient-accumulation-steps", "8",
        "--max-length", "2048",
        "--learning-rate", "2e-4",
        "--lora-rank", "16", "--lora-alpha", "32",
        "--load-in-4bit",
        "--precision", "bf16",
        "--report-to", "none",
        "--artifact-bucket", BUCKET_ID,
        "--run-id", ALL_RUN_ID,
        "--artifact-stage", slug,
    ]
    subprocess.run(command, check=True)
    import json
    report = json.loads((Path(output_dir) / "reports" / "metrics_summary.json").read_text(encoding="utf-8"))
    results[slug] = report

print("\n=== ALL SFT RESULTS ===")
for slug, report in results.items():
    print(f"{slug}: {report}")

In [ ]:
from IPython.display import SVG, display

for slug in MODELS:
    REPORTS_DIR = Path(f"outputs/{slug}-sft") / "reports"
    charts = sorted(REPORTS_DIR.glob("*.svg"))
    print(f"[{slug}] hf://buckets/{BUCKET_ID}/runs/{ALL_RUN_ID}/{slug}/reports")
    for chart in charts:
        display(SVG(filename=str(chart)))